# Only once per runtime

## Load dataset on content from drive

In [1]:
# Wether to move DB from drive to local disk
MOVE_DB_FROM_DRIVE = False

In [2]:
if MOVE_DB_FROM_DRIVE:
  import os
  from huggingface_hub import snapshot_download, login
  from google.colab import userdata

  HF_TOKEN = userdata.get('HF_TOKEN')
  os.environ['HF_TOKEN'] = HF_TOKEN
  login(HF_TOKEN)

  # Define your credentials and repo details
  REPO_ID = "asarra/wikipedia-vector-db"

  # Target local directory where the database will sit unzipped
  LOCAL_DIR = "/content/db_local_extracted"

  print("--> Downloading unzipped vector database from Hugging Face...")
  print("Estimated duration: ~7 minutes (No manual extraction needed!)")

  # snapshot_download downloads files in parallel and displays a native progress bar
  snapshot_download(
      repo_id=REPO_ID,
      repo_type="dataset",
      token=HF_TOKEN,
      local_dir=LOCAL_DIR,
      allow_patterns="db_local/*",
      local_dir_use_symlinks=False  # Replicates the actual files into the folder cleanly
  )

  # Match the exact folder name structure inside your repo if needed
  VECTOR_DB_PATH = os.path.join(LOCAL_DIR, "db_local")

else:
  PARQUET_PATH = '/content/drive/MyDrive/Progetto-NLP/Branch-rag/collection_ita.parquet'
  VECTOR_DB_PATH = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita/db/db_local"
  VECTOR_DB_PATH_LOCAL = "/content/db"
  DS_PATH="/content/drive/MyDrive/Progetto-NLP/Branch-rag/"
  CACHE_DIR = "/content/drive/MyDrive/Progetto-NLP/hf_cache/"

In [3]:
VECTOR_DB_PATH = "/content/db_local_extracted/db_local"

## Installs

In [4]:
  #old installations

#!pip install -q "transformers==4.46.3" sentence-transformers ddgs tqdm lancedb "gptqmodel==1.5.1" "autoawq<0.2.9" "protobuf<6.0" "numpy<2.1"


#Second type of installation (requires restarting the notebook):
!pip install tqdm lancedb
!pip install gptqmodel
!pip install ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.8/334.8 kB 34.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.1/979.1 kB 21.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.8/121.8 kB 14.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 107.5 MB/s eta 0:00:00


In [5]:
# import os
# from google.colab import drive
# drive.mount('/content/drive')

# # 1. Definiamo sia la cache locale temporanea che la destinazione finale su Drive
# LOCAL_CACHE = "/content/local_cache"
# DRIVE_CACHE = "/content/drive/MyDrive/Progetto-NLP/pip_cache"

# os.makedirs(LOCAL_CACHE, exist_ok=True)
# os.makedirs(DRIVE_CACHE, exist_ok=True)

# # 2. Compila TUTTO sul disco locale di Colab (veloce e senza errori di Drive)
# !pip wheel tqdm lancedb gptqmodel ddgs \
#     --wheel-dir {LOCAL_CACHE}

# # 3. Solo ora copiamo i file pronti e compilati su Google Drive in un colpo solo
# !cp -r {LOCAL_CACHE}/* {DRIVE_CACHE}/

# print("✅ Compilazione e backup su Drive completati senza errori!")

In [6]:
# import os
# from google.colab import drive
# drive.mount('/content/drive')

# DRIVE_CACHE = "/content/drive/MyDrive/Progetto-NLP/pip_cache"

# # Usiamo "file:" per prevenire il bug del percorso ignorato su Drive
# !pip install \
#     --no-index \
#     --find-links file:{DRIVE_CACHE} \
#     tqdm lancedb gptqmodel ddgs \
#     --quiet

# Every session

## Connection to PoliMilionaire

### Imports

In [1]:
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from google.colab import userdata, drive
from huggingface_hub import login
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import Callable
import os
import pandas as pd
from transformers import AutoModelForSeq2SeqLM,AutoModelForCausalLM, AutoTokenizer, pipeline
import sys
import time
from typing import Callable
from sentence_transformers import CrossEncoder
from transformers import BitsAndBytesConfig
from sentence_transformers import util
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM
import tqdm
from datasets import load_dataset, load_from_disk
import lancedb
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS
import threading
import concurrent.futures


### Connections

#### Google Drive

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


#### Hugging Face

In [3]:
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
login(HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


#### Game APIs

In [4]:
repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

if os.path.exists("../"+repo_name):
    print("Repository already present, update...")
    !git pull
else:
    print("Repository clone...")
    !git clone {repo_url}
    %cd {repo_name}

sys.path.append('/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/NLP_assignment_api_client')

from millionaire_client import MillionaireClient, AuthenticationError, GameError

API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'GliEmbeddingRuspanti'
PASSWORD = 'GliEmbeddingRuspanti'

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as: {user.username} (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

Repository clone...
Cloning into 'NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi'...
remote: Enumerating objects: 340, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 340 (delta 29), reused 24 (delta 24), pack-reused 304 (from 2)
Receiving objects: 100% (340/340), 35.65 MiB | 18.06 MiB/s, done.
Resolving deltas: 100% (165/165), done.
/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi
Logged in as: GliEmbeddingRuspanti (role: student)


#### Load dataset on content from drive

In [5]:
# !cp -r /content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita/db/db_local.zip /content/db_local
# !unzip /content/db_local -d /content/db_local_extracted
# VECTOR_DB_PATH = "/content/db_local_extracted/db_local"


### Model class

In [6]:
class Model:
    """Base class. Subclasses implement generate().
       answer_fn decide how to get the final option."""
    def __init__(self, name: str, answer_fn: Callable):
        self.name = name
        self.answer_fn = answer_fn

    def generate(self, question: str, system_prompt: str = "") -> str:
        raise NotImplementedError

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        raw_output = self.generate(question, system_prompt)
        summary_answer, answer = self.answer_fn(raw_output, options)
        print(f"MODEL ANSWER ----->{raw_output}")
        return summary_answer, answer

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r}, answer_fn={self.answer_fn.__name__!r})"


### Creating a dataset for benchmark

In [7]:
DATASET_PATH = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/correct_questions.jsonl"

def save_correct_question(game, question, answer_id, chosen_answer):
    with open(DATASET_PATH, "r", encoding="utf-8") as f:
      lines = f.readlines()
      for line in lines:
        data = json.loads(line)
        if (data["question"]==question):
          return

    entry = {
        "competition": game.state.competition.id,
        "question": question.text,
        "options": [opt.text for opt in question.options],
        "correct_answer_id": answer_id,
        "correct_answer": chosen_answer
    }

    with open(DATASET_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

### The Game

In [8]:
import json
def play_game(game, model, sys_prompt):
  log = []
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      print()

      for opt in question.options:
          print(f"  [{opt.id}] {opt.text}")

      time_left = game.time_remaining
      if time_left:
          print(f"\nTime remaining: {time_left:.1f}s")

      options = {f"{opt.id}": opt.text for opt in question.options}

      t0 = time.time()
      answer_summary, answer_input = model.answer(question.text, options, sys_prompt)
      inference_time = time.time() - t0
      print(f"Model answer: {answer_input}")
      answer_id = int(answer_input)

      choosen_answer = question.options[answer_id]

      result = game.answer(answer_id)

      if result.correct:
          print(" CORRECT!")
          save_correct_question(game, question, answer_id, choosen_answer.text)
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

      # Log the outcome
      entry = {
          'competition'     : game.state.competition.id,
          'level'           : game.current_level,
          'question'        : question.text,
          'options'         : [str(opt.text) for opt in question.options],
          'chosen_option'   : choosen_answer.text,
          'correct'         : result.correct,
          'timed_out'       : result.timed_out,
          'inference_time'  : round(inference_time, 2),
          'complete_prompt' : model.complete_prompt,
          'answer_summary'  : answer_summary,
      }
      log.append(entry)

  print("========SAVING=======")
  with open("/content/drive/MyDrive/Progetto-NLP/Branch-rag/log.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(log) + "\n")

  summary = {
        'model'           : model.name,
        'final_level'     : game.current_level,
        'earned_amount'   : game.earned_amount,
        'num_questions'   : len(log),
        'num_correct'     : sum(1 for e in log if e['correct']),
        'num_timed_out'   : sum(1 for e in log if e['timed_out']),
        'avg_inference_s' : round(sum(e['inference_time'] for e in log) / max(len(log), 1), 2),
        'log'             : log,
    }

  print("\n=== Game Summary ===")
  print(f"Competition: {game.state.competition.id}")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")

  return summary

### RAG model class

#### Web Retireval

In [9]:
def _scrape_page(url: str, max_chars: int = 3000, stop_event: threading.Event | None = None) -> str:
    if stop_event and stop_event.is_set():
        return ""
    try:
        r = requests.get(url, timeout=5, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        return soup.get_text(separator=" ", strip=True)[:max_chars]
    except Exception:
        return ""

def _fetch_web_documents(query: str, k: int = 10, full_text: bool = True, stop_event: threading.Event | None = None,) -> list[str]:
    """Cerca su DuckDuckGo e restituisce una lista di testi (snippet o pagine complete)."""
    docs = []
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=k))

    for r in results:
        if full_text:
            text = _scrape_page(r["href"])
            docs.append(text if text else r["body"])  # fallback sullo snippet
        else:
            docs.append(r["body"])  # solo snippet (~200 char)

    return docs

#### Load model

In [10]:
def load_rag_model(use_db: bool = True):
  bi_enc = SentenceTransformer('BAAI/bge-m3', model_kwargs={"torch_dtype": torch.float16})
  reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=1024)
  model_id = "Qwen/Qwen2.5-7B-Instruct"
  tokenizer = AutoTokenizer.from_pretrained(model_id)
  model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
  )
  collection = ds = None
  if use_db:
    # Use a local variable to manage the path within this function
    db_path_for_use = VECTOR_DB_PATH # Initialize with the global value

    if not os.path.exists(db_path_for_use):
      # If the primary path doesn't exist, use the fallback
      db_path_for_use = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/embedding_collection_ita/db/db_local/"
      print(f"Warning: Original VECTOR_DB_PATH not found. Using fallback path: {db_path_for_use}")

    table_name = "wiki_rag_collection"
    if not os.path.exists(db_path_for_use):
      print("ERRORE CRITICO: La cartella base non esiste per Colab!")
    else:
        contenuto = os.listdir(db_path_for_use)
        print(f"Cosa c'è fisicamente dentro '{db_path_for_use}':")
        print(contenuto)

        if f"{table_name}.lance" in contenuto:
            print(f"\n✅ PERFETTO! La tabella fisica '{table_name}.lance' C'È.")
        else:
            print(f"\n❌ ERRORE! La tabella fisica '{table_name}.lance' MANCA in questa cartella.")
            print("Probabilmente il path è sbagliato o la cartella è nidificata più a fondo.")

    print("\n--- TEST LANCEDB ---")
    db = lancedb.connect(db_path_for_use)

    tabelle_presenti = db.table_names()
    print(f"Tabelle viste da LanceDB: {tabelle_presenti}")

    if table_name in tabelle_presenti:
        collection = db.open_table(table_name)
        print(f"🎉 SUCCESSO! Tabella '{table_name}' aperta.")
    else:
        print(f"❌ FALLIMENTO. LanceDB non vede la tabella.")

    #TODO:Move those path definitions outside (but check if can read it, as done for VECTOR_DB)
    # Assuming DS_PATH (directory for Arrow cache) and PARQUET_PATH are defined earlier
    DS_PATH="/content/drive/MyDrive/Progetto-NLP/Branch-rag/"
    PARQUET_PATH = '/content/drive/MyDrive/Progetto-NLP/Branch-rag/collection_ita.parquet'

    # We save Hugging Face datasets as a directory structure, not a single file
    ds_arrow_dir = os.path.join(DS_PATH, "ds_embedding_collection_ita")

    # Attempt to load from native Hugging Face Disk Cache (Super Fast Arrow Format)
    if os.path.exists(ds_arrow_dir):
        print("Attempting to load dataset from native disk cache...")
        try:
            ds = load_from_disk(ds_arrow_dir)
            _ = len(ds)  # Quick verification
            print("Dataset loaded successfully from disk cache.")
        except Exception as e:
            print(f"Failed to load dataset from cache ({e}). Attempting to load from raw Parquet instead.")
            ds = None

    # Fallback: If cache doesn't exist or is corrupted, load from Parquet
    if ds is None:
        if os.path.exists(PARQUET_PATH):
            print("Loading dataset from Parquet...")
            ds = load_dataset("parquet", data_files=PARQUET_PATH, split="train")
            print("Dataset loaded successfully from Parquet.")

            # Save it natively to disk for blazing fast future loading
            print("Caching dataset to disk for future use...")
            ds.save_to_disk(ds_arrow_dir)
            print("Dataset cached successfully.")
        else:
            print(f"Error: Neither cache directory ({ds_arrow_dir}) nor Parquet file ({PARQUET_PATH}) found.")
            raise FileNotFoundError(f"Cannot load dataset. Parquet file not found at {PARQUET_PATH}")

  return bi_enc, reranker, model, tokenizer, collection, ds



#### Query processing

In [11]:
def translate_and_rewrite_query(self, query):
    # 1. Prompt "Aggressivo" per Traduzione e Ottimizzazione
    system_prompt = (
        "Sei un traduttore esperto e un ottimizzatore per motori di ricerca. "
        "Il tuo compito è tradurre la domanda dell'utente in ITALIANO e riscriverla "
        "in modo chiaro e descrittivo, ideale per cercare in un database di Wikipedia in italiano. "
        "REGOLE FONDAMENTALI:\n"
        "1. L'output deve essere ESCLUSIVAMENTE in lingua italiana.\n"
        "2. Restituisci SOLO la domanda tradotta e ottimizzata.\n"
        "3. Non aggiungere saluti, spiegazioni, 'Ecco la traduzione' o virgolette."
    )

    user_prompt = f"Traduci e ottimizza la seguente domanda:\n<domanda>\n{query}\n</domanda>\n\nDomanda in italiano:"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # Trasformazione in tensori
    inputs = self.tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(self.model.device)

    # 2. Generazione (Max 60 token, niente creatività)
    outputs = self.model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        use_cache=True,
        pad_token_id=self.tokenizer.eos_token_id
    )

    torch.cuda.synchronize()

    # Decodifica
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    query_ita = self.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Pulizia extra
    query_ita = query_ita.replace('"', '').replace("'", "")

    return query_ita

#### Query answering logic

In [13]:

TIMEOUT = 8

def rag_sota(self, query, options_text):
    start_time = time.time()
    print("\nInizio retrieval...")
    db_time = web_time = db_lance_time = start_lance_time = start_time

        # A1. RETRIEVAL (LanceDB)
    retrieved_docs = []
    if self.use_db:
      #processed_query = translate_and_rewrite_query(self, query)
      processed_query = query
      #torch.cuda.empty_cache()
      query_vector = self.bi_enc.encode([processed_query]).tolist()[0]
      start_lance_time = time.time()
      risultati = self.collection.search(query_vector).limit(20).to_pandas()
      db_lance_time = time.time()

      db_docs = []
      # B. PREPARAZIONE DOCUMENTI
      for _, row in risultati.iterrows():
            doc_id = int(row['id'])
            content = self.ds[doc_id]['content'].strip()
            if len(content) > 50:
              db_docs.append({
                  "content": content,
                  "source": "wikipedia"
              })


      retrieved_docs.extend(db_docs)
      db_time = time.time()

    # A2. RETRIEVAL (Web)
    if self.use_web:
      if not self.use_db:
        TIMEOUT = 15

      web_raw = []
      stop_event = threading.Event()
      executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
      try:
        future = executor.submit(
          _fetch_web_documents,
          f"{query}",
          self.top_k_web,
          self.full_text,
          stop_event,
        )
        web_raw = future.result(timeout=TIMEOUT)
        print(f"Recuperati {len(web_raw)} documenti dal web.")
      #       # Creiamo un Executor temporaneo per lanciare la funzione
      #       with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
      #           # 'submit' lancia la funzione in background passando i suoi argomenti
      #           query_web = f"{query}"
      #           future = executor.submit(_fetch_web_documents, query_web, self.top_k_web, self.full_text)

      #           # 'result' aspetta la risposta, ma SOLO per i secondi indicati!
      #           web_raw = future.result(timeout=TIMEOUT)


      except concurrent.futures.TimeoutError:
          stop_event.set()
          print(f"Timeout Web: DuckDuckGo ha superato i {TIMEOUT} secondi. Ignorato.")
      except Exception as e:
          stop_event.set()
          print(f"Errore Web imprevisto: {e}. Ignorato.")
      finally:
        executor.shutdown(wait=False)
      # Se ha fallito o è andato in timeout, web_raw sarà vuoto [], il ciclo lo gestirà tranquillamente
      web_docs = [{"content": doc, "source": "web"} for doc in web_raw]
      retrieved_docs.extend(web_docs)
      web_time = time.time()

    # C. RERANKING
    couples = [[query, doc["content"]] for doc in retrieved_docs]
    scores = self.reranker.predict(couples)

    docs_with_score = list(zip(scores, retrieved_docs))
    docs_with_score.sort(key=lambda x: x[0], reverse=True)

    if self.debug:
      for rank, (score, doc) in enumerate(docs_with_score, start=1):

        print(f"[{rank}] score={score:.4f}")

        if isinstance(doc, dict):
            print(doc.get("content", "")[:300])

        else:
            print(str(doc)[:300])

        print("=" * 80)

    # D. TOP K DOCS (con TRUNCATION DI SICUREZZA)
    top_docs_objects = [doc for score, doc in docs_with_score[:self.top_k_context]]
    docs_context = "\n\n---\n\n".join([doc["content"] for doc in top_docs_objects])

    sources = [doc["source"] for doc in top_docs_objects]
    print(f"Sorgenti nei Top K: {sources}")

    if len(docs_context) > 8000:
        docs_context = docs_context[:8000] + "\n... [TRONCATO PER SICUREZZA]"
        print("TRONCATO")

    print("Retrieval finito.")
    end_time_retrivial = time.time()

    # E. PULIZIA RAM
    try:
      del risultati
    except NameError:
      pass
    del retrieved_docs
    del couples
    torch.cuda.empty_cache()
    system_prompt = "You are a quiz solver. Read the context and answer the question. Do not use external knowledge."

    user_prompt = f"""<context>
{docs_context}
</context>

Question: {query}
Options: {options_text}

Instructions: The context can be in Italian or in English, the options are in English. Analyze the context, reason on its meaning and write ONLY the ID of the correct option.
Answer:"""


    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt_testo = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # MODIFICA 1: Chiamiamo la variabile 'inputs' e aggiungiamo return_dict=True
    inputs = self.tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(self.model.device)

    # G. INFERENZA NATIVA ULTRA-VELOCE
    outputs = self.model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        use_cache=True,
        pad_token_id=self.tokenizer.eos_token_id
    )

    # MODIFICA 3: Dobbiamo prendere la lunghezza da inputs['input_ids']
    generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    predicted_answer = self.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    # Se non è un numero valido tra 0 e 3, scegli un numero casuale
    if not predicted_answer.isdigit() or int(predicted_answer) not in range(4):
        predicted_answer = rag_fallback(self, query, options_text)

    end_time = time.time()
    tempo_lance = db_lance_time - start_lance_time
    tempo_esecuzione = end_time - start_time
    tempo_retrivial = end_time_retrivial - start_time
    tempo_db = db_time - start_time
    tempo_web = web_time - db_time
    print(f"Tempo lance: {tempo_lance:.2f} secondi")
    print(f"Tempo retrivial: {tempo_retrivial:.2f} secondi")
    print(f"Tempo retrivial db: {tempo_db:.2f} secondi")
    print(f"Tempo retrivial web: {tempo_web:.2f} secondi")
    print(f"Tempo di esecuzione totale: {tempo_esecuzione:.2f} secondi")

    return prompt_testo, predicted_answer, top_docs_objects

In [14]:
import random

def rag_fallback(self, query, options_text):
  system_prompt = "You are a quiz solver. Use your knowledge to answer the question"

  user_prompt = f"""

Question: {query}
Options: {options_text}

Instructions: Analyze the context, reason on its meaning and write ONLY the ID of the correct option.
Answer:"""


  messages = [
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": user_prompt}
  ]

  prompt_testo = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

  # MODIFICA 1: Chiamiamo la variabile 'inputs' e aggiungiamo return_dict=True
  inputs = self.tokenizer.apply_chat_template(
      messages,
      tokenize=True,
      add_generation_prompt=True,
      return_tensors="pt",
      return_dict=True
  ).to(self.model.device)

  # G. INFERENZA NATIVA ULTRA-VELOCE
  outputs = self.model.generate(
      **inputs,
      max_new_tokens=10,
      do_sample=False,
      use_cache=True,
      pad_token_id=self.tokenizer.eos_token_id
  )

  # MODIFICA 3: Dobbiamo prendere la lunghezza da inputs['input_ids']
  generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
  predicted_answer = self.tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
  if not predicted_answer.isdigit() or int(predicted_answer) not in range(4):
      predicted_answer = str(random.randint(0, 3))

  return predicted_answer



In [15]:
torch.cuda.empty_cache()


#### RAG model class definition

In [16]:
class RAGModel(Model):
  def __init__(self, name: str, use_db: bool = True, use_web: bool = True, top_k_web: int = 20, full_text: bool = True, debug = False, top_k_context = 3):
    self.name=name
    self.use_db=use_db
    self.use_web=use_web
    self.top_k_web=top_k_web
    self.full_text=full_text
    self.debug = debug
    self.top_k_context = top_k_context
    self.bi_enc, self.reranker, self.model, self.tokenizer, self.collection, self.ds = load_rag_model(use_db)

  def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
    complete_prompt , generated_answer, _ = rag_sota(self, question, options_text=options)
    self.complete_prompt = complete_prompt
    self.generated_answer = generated_answer
    return "", generated_answer

## Instantiate model

In [17]:
rag_model = RAGModel("RAG", use_db=False, top_k_web=10)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [18]:
rag_model.top_k_context = 2

In [19]:
models = {
    "RAG": {"model": rag_model, "system_prompt": ""}}

## Play and print results

In [20]:
results = {}

for model_name, config in models.items():
    print(f"\n########## MODEL: {model_name} ##########")

    model = config["model"]
    system_prompt = config["system_prompt"]

    model_results = []

    for comp_id in [0, 1, 2, 3, 4, 5]:
        print(f"\n--- Competition {comp_id} ---")

        game = client.game.start(competition_id=comp_id)

        summary = play_game(game, model, system_prompt)

        model_results.append(summary)

    results[model_name] = model_results


########## MODEL: RAG ##########

--- Competition 0 ---

--- Level 1 ---
Q: Which director's influence can be seen in Spielberg's use of wide-angle lenses and handheld camera work in his films?

  [0] Frank Capra
  [1] John Ford
  [2] Stanley Kubrick
  [3] Ingmar Bergman

Time remaining: 29.9s

Inizio retrieval...
Recuperati 10 documenti dal web.
Sorgenti nei Top K: ['web', 'web']
Retrieval finito.
Tempo lance: 0.00 secondi
Tempo retrivial: 14.70 secondi
Tempo retrivial db: 0.00 secondi
Tempo retrivial web: 12.12 secondi
Tempo di esecuzione totale: 20.68 secondi
Model answer: 1
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
Q: In which U.S. state was the film 'The Shawshank Redemption' primarily shot?

  [0] Maine
  [1] California
  [2] New York
  [3] Ohio

Time remaining: 27.9s

Inizio retrieval...
Recuperati 10 documenti dal web.
Sorgenti nei Top K: ['web', 'web']
Retrieval finito.
Tempo lance: 0.00 secondi
Tempo retrivial: 7.08 secondi
Tempo retrivial db: 0.00 secondi
Tempo ret

In [21]:
def print_results(results):
    for model_name, competitions in results.items():

        print("\n" + "=" * 80)
        print(f"MODELLO: {model_name}")
        print("=" * 80)

        for i, summary in enumerate(competitions):

            print(f"\n🏁 Competition {i}")
            print("-" * 60)

            print(f"Model name        : {summary['model']}")
            print(f"Final level       : {summary['final_level']}")
            print(f"Earned amount     : €{summary['earned_amount']}")
            print(f"Questions         : {summary['num_questions']}")
            print(f"Correct answers   : {summary['num_correct']}")
            print(f"Timed out         : {summary['num_timed_out']}")
            print(f"Avg inference     : {summary['avg_inference_s']} s")

            accuracy = (
                summary['num_correct'] / summary['num_questions'] * 100
                if summary['num_questions'] > 0 else 0
            )

            print(f"Accuracy          : {accuracy:.1f}%")

            print("\n📋 Question Log")
            print("-" * 60)

            confidence_array = []

            for q_idx, entry in enumerate(summary['log'], start=1):

                status = "✅" if entry['correct'] else "❌"

                if entry.get('timed_out'):
                    status = "⏰"

                # ─────────────────────────────────────────────
                # CONFIDENCE EXTRACTION (robust fallback chain)
                # ─────────────────────────────────────────────
                answer_summary = entry.get("answer_summary", {})

                if isinstance(answer_summary, dict):
                    if "normalized_margin" in answer_summary:
                        conf = answer_summary["normalized_margin"]

                    elif "confidence" in answer_summary:
                        conf = answer_summary["confidence"]

                    else:
                        conf = None
                else:
                    conf = None

                confidence_array.append(conf)

                print(
                    f"{q_idx:02d}. "
                    f"{status} "
                    f"Time: {entry['inference_time']:.2f}s "
                    f"Conf: {conf if conf is not None else 'N/A'}"
                )

            # ─────────────────────────────────────────────
            # PRINT SUMMARY CONFIDENCE ARRAY
            # ─────────────────────────────────────────────
            print("\n📊 Confidence Array:")
            if any(c is not None for c in confidence_array):
                print(confidence_array)
            else:
                print("Confidence not available")

        print("\n")

# final print
print_results(results)


MODELLO: RAG

🏁 Competition 0
------------------------------------------------------------
Model name        : RAG
Final level       : 5
Earned amount     : €500
Questions         : 5
Correct answers   : 4
Timed out         : 0
Avg inference     : 17.21 s
Accuracy          : 80.0%

📋 Question Log
------------------------------------------------------------
01. ✅ Time: 20.70s Conf: N/A
02. ✅ Time: 12.14s Conf: N/A
03. ✅ Time: 16.03s Conf: N/A
04. ✅ Time: 21.68s Conf: N/A
05. ❌ Time: 15.51s Conf: N/A

📊 Confidence Array:
Confidence not available

🏁 Competition 1
------------------------------------------------------------
Model name        : RAG
Final level       : 6
Earned amount     : €1000
Questions         : 6
Correct answers   : 5
Timed out         : 1
Avg inference     : 23.23 s
Accuracy          : 83.3%

📋 Question Log
------------------------------------------------------------
01. ✅ Time: 14.23s Conf: N/A
02. ✅ Time: 22.70s Conf: N/A
03. ✅ Time: 18.62s Conf: N/A
04. ✅ Time: 28.

# Test

In [22]:
def test_game(dataset, model, sys_prompt = ""):
  model_summary = {}
  for current_competition in range(0, 5):
    print(f"\n--- Competition {current_competition} ---")
    current_level = 1
    model_summary[current_competition] = {
        "max_level": 1,
        "state": "Success"
    }

    query_competition = [entry for entry in dataset if entry["competition"] == current_competition]

    if not query_competition:
        model_summary[current_competition]["state"] = "No question present"
        continue

    for entry in query_competition:
      question = entry["question"]
      options = entry["options"]
      correct_answer_id = entry["correct_answer_id"]

      initial_time = time.time()
      answer_summary, answer_input = model.answer(question, options, sys_prompt)
      end_time = time.time()

      print(f"Question: {question}, options: {options}")
      print(f"Model answer: {answer_input}, correct answer: {correct_answer_id}")


      if (end_time - initial_time) > 30:
        print("TIMED OUT, GAME OVER")
        model_summary[current_competition]["state"] = "Timed Out"
        break

      if (answer_input != correct_answer_id):
        print("WRONG ANSWER, GAME OVER")
        model_summary[current_competition]["state"] = "Wrong answer"
        break

      model_summary[current_competition]["max_level"] = current_level
      current_level += 1
  return model_summary


In [23]:
import json
DATASET_PATH = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/correct_questions.jsonl"

def test_models(models):
  dataset = []
  with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        dataset.append(json.loads(line))
  summary = {}
  for model_name, config in models.items():
    model = config["model"]
    system_prompt = config["system_prompt"]

    model_summary = test_game(dataset, model, system_prompt)
    summary[model_name] = model_summary
  return summary

In [24]:
test_models(models)


--- Competition 0 ---

Inizio retrieval...
Recuperati 10 documenti dal web.
Sorgenti nei Top K: ['web', 'web']
Retrieval finito.
Tempo lance: 0.00 secondi
Tempo retrivial: 7.89 secondi
Tempo retrivial db: 0.00 secondi
Tempo retrivial web: 6.13 secondi
Tempo di esecuzione totale: 13.41 secondi
Question: Which of the following films did Robert De Niro win an Academy Award for Best Actor?, options: ['The Godfather Part II', 'Taxi Driver', 'Raging Bull', 'The Deer Hunter']
Model answer: 3, correct answer: 2
WRONG ANSWER, GAME OVER

--- Competition 1 ---

Inizio retrieval...
Recuperati 10 documenti dal web.
Sorgenti nei Top K: ['web', 'web']
Retrieval finito.
Tempo lance: 0.00 secondi
Tempo retrivial: 6.12 secondi
Tempo retrivial db: 0.00 secondi
Tempo retrivial web: 3.97 secondi
Tempo di esecuzione totale: 12.56 secondi
Question: What term describes the period when ancient Egypt was divided into smaller dynasties, between the end of the Middle Kingdom and the start of the New Kingdom?, op

{'RAG': {0: {'max_level': 1, 'state': 'Wrong answer'},
  1: {'max_level': 1, 'state': 'Wrong answer'},
  2: {'max_level': 1, 'state': 'Wrong answer'},
  3: {'max_level': 1, 'state': 'Wrong answer'},
  4: {'max_level': 1, 'state': 'Wrong answer'}}}